## Evaluating Hyperparameters

#### This script follows the structure below

## 1.Importing libraries
## 2.Data Wrangling
## 3.Reshape for running the Model
## 4.Data Split
## 5. Hyperparameter Optimization - Random Forest 2010s all Weather Stations
### 5.1 Grid Search
### 5.2 Random Search
## 6. Running Random Forest Model with Optimized Search Parameters of the RANDOM Search
## 7. Uncovering Feature Importance
## 8.Hyperparameter Optimization - Random Forest Budapest all years

# 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import operator
import time
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn import datasets  
from sklearn.ensemble import RandomForestClassifier
from numpy import argmax
from sklearn.model_selection import train_test_split
from sklearn import metrics  
from sklearn.tree import plot_tree
from sklearn import tree
from sklearn.model_selection import GridSearchCV
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [ ]:
# Creating a path for importing the climate data set

path = r'/Users/daniel/Desktop/Ordner/Data Analyst/Data Analytics Course/Data Specialization/Data Sets'

In [ ]:
# Import the data set

df_weather = pd.read_csv(os.path.join(path, 'weather_clean.csv'),index_col = False)

df_answer = pd.read_csv(os.path.join(path, 'Dataset-Answers-Weather_Prediction_Pleasant_Weather.csv'),index_col = False)

# 2. Data Wrangling

In [ ]:
# Show all the columns in the data set
pd.set_option('display.max_columns', 200)

# Show all the rows in the data set
pd.set_option('display.max_rows', None)

#### Dataset Weather

In [ ]:
# Check for correct import 

df_weather.head()

In [ ]:
# Check for shape

df_weather.shape

In [ ]:
# Check for missing values

df_weather.isnull().sum()

In [ ]:
# Check for duplicate values

duplicate_value = df_weather.duplicated()

print(f"Total duplicate value: {duplicate_value.sum()}")

In [ ]:
# Check the basic statistic values
df_weather.describe()

#### Dataset Answers

In [ ]:
# Check for correct import 

df_answer.head()

In [ ]:
# Check for the shape

df_answer.shape

In [ ]:
# Check for missing values

df_answer.isnull().sum()

In [ ]:
# Check for duplicate values

duplicate_value = df_answer.duplicated()

print(f"Total duplicate value: {duplicate_value.sum()}")

#### Reduce data to one decade. Chosen decade: 2010 - 2019

In [ ]:
# Reduce observations dataset to 2010's

df_decade = df_weather[(df_weather['DATE'].astype(str).str[:4] >= '2010') & (df_weather['DATE'].astype(str).str[:4] <= '2019')]
df_decade

In [ ]:
# Reduce answers dataset to 2010's

answers_decade = df_answer[(df_answer['DATE'].astype(str).str[:4] >= '2010') & (df_answer['DATE'].astype(str).str[:4] <= '2019')]
answers_decade

In [ ]:
# Extract stations list

stations = [col.split('_')[0] for col in df_decade.columns if '_' in col]

In [ ]:
# Create a set of unique station names

unique_stations = set(stations)
unique_stations

In [ ]:
# Create a dictionary to store the frequency of entries for each station
station_frequencies = {}

for station in unique_stations:
    # Select columns that belong to the current station
    station_columns = [col for col in  df_decade.columns if col.startswith(station)]
    
    # Count non-missing entries across all columns for the station
    station_frequencies[station] =  df_decade[station_columns].notna().sum().sum()

# Print the frequency of entries for each station
print("Frequency of entries for each weather station:")
for station, freq in station_frequencies.items():
    print(f"{station}: {freq} entries")

In [ ]:
# Drop unnecessary columns

df_decade.drop(['DATE', 'MONTH'], axis=1, inplace=True)
df_decade.head()

In [ ]:
df_decade.shape # observations dataset has the correct shape

In [ ]:
answers_decade.drop(columns = 'DATE', inplace = True)

In [ ]:
answers_decade.shape # predictions dataset has the correct shape

# 3. Reshaping for running the Model

#### The final shapes should be X = (3652, 135) and y = (3652,) for one decade of information.

In [ ]:
X = df_decade

In [ ]:
y = answers_decade

In [ ]:
# Turn X and y from a df to arrays

X = np.array(X)
y = np.array(y)

In [ ]:
X.shape

In [ ]:
y.shape

# 4. Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

# 5. Hyperparameter Optimization - Random Forest 2010s all Weather Stations

### 5.1 Grid Search

In [ ]:
# Create a RF classifier

clf = RandomForestClassifier()

In [ ]:
grid_space = {
    'max_depth': [3, 10, None],  # Focus on a smaller range
    'n_estimators': [50, 100],  # Reduced number of estimators
    'max_features': [15, 50],  # Focus on fewer values
    'min_samples_leaf': [1, 2],  # Smaller range
    'min_samples_split': [2, 3]  # Avoid using 1 (invalid in sklearn)
}

In [ ]:
start = time.time()
grid = GridSearchCV(clf, param_grid=grid_space, cv=3, scoring='accuracy', verbose=3, n_jobs=-1)
model_grid = grid.fit(X_train, y_train)
print('Search took %s minutes' % ((time.time() - start)/60))

In [ ]:
# Print grid search results

print('Best GRID search hyperparameters are: '+str(model_grid.best_params_))
print('Best GRID search score is: '+str(model_grid.best_score_))

### 5.2 Random Search

In [ ]:
# Define random search cv

rs_space={'max_depth':list(np.arange(10, 100, step=10)) + [None],
              'n_estimators':np.arange(10, 500, step=50),
              'max_features':randint(15, 135),
              'criterion':['gini','entropy'],
              'min_samples_leaf':randint(1,4),
              'min_samples_split':np.arange(2, 10, step=2)
         }

In [ ]:
# Create a RF classifier

clf2= RandomForestClassifier()

In [ ]:
start = time.time()
rf_random = RandomizedSearchCV(clf2, rs_space, n_iter=10, scoring='accuracy', verbose=3, n_jobs=-1, cv=3)
model_random = rf_random.fit(X_train, y_train)
print('Search took %s minutes' % ((time.time() - start)/60))

In [ ]:
# Random random search results

print('Best RANDOM search hyperparameters are: '+str(model_random.best_params_))
print('Best RANDOM search score is: '+str(model_random.best_score_))

In [ ]:
# Grid search results vs.

print('Best GRID search hyperparameters are: '+str(model_grid.best_params_))
print('Best GRID search score is: '+str(model_grid.best_score_))

# Random random search results

print('Best RANDOM search hyperparameters are: '+str(model_random.best_params_))
print('Best RANDOM search score is: '+str(model_random.best_score_))

__Comment:__ The **random** seach score is slightly higher with **63.9%** compared to the **grid** seach score with **63,8%**

# 6. Running Random Forest Model with Optimized Search Parameters of the RANDOM Search

In [ ]:
# Create a RF classifier with the best results from Random Search
clf3 = RandomForestClassifier(
    n_estimators=460,
    max_depth=None,
    max_features=50,
    min_samples_leaf=2,
    min_samples_split=6,
    criterion='gini'
)

# Training the model on the training dataset
clf3.fit(X_train, y_train)

In [ ]:
# Perform predictions on the test dataset
y_pred = clf3.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
fig1 = plt.figure(figsize=(80,40))
plot_tree(clf3.estimators_[15], fontsize = 20, feature_names = df_decade.columns, class_names=['unpleasant', 'pleasant'], filled=True);

fig1.savefig(os.path.join(path, 'Visualizations','Random Forest 10s Optimized.png'),bbox_inches='tight')

# 7. Uncovering Feature Importance

In [ ]:
# Retrieve feature importances from the trained model

newarray = clf3.feature_importances_
print(clf3.feature_importances_.shape)
newarray

In [ ]:
# Reshape newarray

newarray = newarray.reshape(-1,15,9)
print(newarray.shape)
newarray

In [ ]:
# Collapse this shape into one observation for each weather station

sumarray = np.sum(newarray[0], axis=1)
sumarray

In [ ]:
# Convert the set of unique stations to a list

unique_stations_list = list(unique_stations)

In [ ]:
important = pd.Series(sumarray, index = unique_stations_list)
important = important.sort_values(ascending = False)
important

In [ ]:
# Create a df to associate weather stations with their importances

df_importance = pd.DataFrame({
    'Weather Station': unique_stations_list,
    'Importance': sumarray
})

df_importance = df_importance.sort_values(by='Importance', ascending = False)

In [ ]:
# Plot the results

%matplotlib inline

plt.style.use('fivethirtyeight')
print(unique_stations_list)

plt.bar(df_importance['Weather Station'], df_importance['Importance'], orientation = 'vertical')
plt.xticks(rotation='vertical')
plt.xlabel('Weather Station')
plt.ylabel('Importance')
plt.title('Weather Station Importance 2010s')

plt.savefig(os.path.join(path, 'Visualizations', 'Station_feature_importances_optimized.png'), bbox_inches='tight')

plt.show()

# 8.Hyperparameter Optimization - Random Forest Budapest all years

In [ ]:
# Create a list of the columns containing "MUNCHENB" in their names
BUDAPEST_list = [col for col in df_weather.columns if 'BUDAPEST' in col]
BUDAPEST_list

In [ ]:
# Create a dataframe with those columns

df_BUDAPEST = df_weather[BUDAPEST_list]
df_BUDAPEST

In [ ]:
# Reduce answers dataset to Basels answers only

answers_BUDAPEST = df_answer['BUDAPEST_pleasant_weather']
answers_BUDAPEST

In [ ]:
df_BUDAPEST.shape # observations dataset has the correct shape

In [ ]:
answers_BUDAPEST.shape # predictions dataset has the correct shape

### Reshaping for modeling

In [ ]:
X3 = df_BUDAPEST

In [ ]:
y3 = answers_BUDAPEST

In [ ]:
# Turn X2 and y2 from df to arrays

X = np.array(X3)
y = np.array(y3)

In [ ]:
X.shape

In [ ]:
y.shape

### Data Split

In [ ]:
# Split data into train and test sets

X_train, X_test, y_train, y_test = train_test_split(X,y,random_state = 42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

In [ ]:
y_test

### Hyperparameter Optimization - Grid Search

In [ ]:
# Create a RF classifier

clf= RandomForestClassifier()

In [ ]:
grid_space = {
    'max_depth': [3, 10, None],  # Focus on a smaller range
    'n_estimators': [50, 100],  # Reduced number of estimators
    'max_features': [15, 50],  # Focus on fewer values
    'min_samples_leaf': [1, 2],  # Smaller range
    'min_samples_split': [2, 3]  # Avoid using 1 (invalid in sklearn)
}

In [ ]:
start = time.time()
grid = GridSearchCV(clf, param_grid=grid_space, cv=3, scoring='accuracy', verbose=3, n_jobs=-1)
model_grid = grid.fit(X_train, y_train)
print('Search took %s minutes' % ((time.time() - start)/60))

In [ ]:
# Print grid search results

print('Best GRID search hyperparameters are: '+str(model_grid.best_params_))
print('Best GRID search score is: '+str(model_grid.best_score_))

### Hyperparameter Optimization - Random Search

In [ ]:
# Define random search cv

rs_space={'max_depth':list(np.arange(10, 100, step=10)) + [None],
              'n_estimators':np.arange(10, 500, step=50),
              'max_features':randint(1,7),
              'criterion':['gini','entropy'],
              'min_samples_leaf':randint(1,4),
              'min_samples_split':np.arange(2, 10, step=2)
         }

In [ ]:
# Create a RF classifier

clf2= RandomForestClassifier()

In [ ]:
start = time.time()
rf_random = RandomizedSearchCV(clf2, rs_space, n_iter=200, scoring='accuracy', verbose=3, n_jobs=-1, cv=3)
model_random = rf_random.fit(X_train, y_train)
print('Search took %s minutes' % ((time.time() - start)/60))

In [ ]:
# Random random search results

print('Best RANDOM search hyperparameters are: '+str(model_random.best_params_))
print('Best RANDOM search score is: '+str(model_random.best_score_))

In [ ]:
# Grid search results vs.

print('Best GRID search hyperparameters are: '+str(model_grid.best_params_))
print('Best GRID search score is: '+str(model_grid.best_score_))

# Random random search results

print('Best RANDOM search hyperparameters are: '+str(model_random.best_params_))
print('Best RANDOM search score is: '+str(model_random.best_score_))

__Comment:__ The search score is **100%** for **Grid** and **Random**. But the **Random** search takes the idea behind a grid seach and makes it a bit more efficient at the risk of possibly missing an ideal combination.

### Running Random Search Hyperparameter Values

In [ ]:
# Create a RF classifier with the best results from Random Search
clf3 = RandomForestClassifier(
    n_estimators=260,
    max_depth=70,
    max_features=4,
    min_samples_leaf=1,
    min_samples_split=4,
    criterion='gini'
)

# Training the model on the training dataset
clf3.fit(X_train, y_train)

In [ ]:
# Perform predictions on the test dataset
y_pred = clf3.predict(X_test)
  
# Use metrics module for accuracy calculation
print("Model Accuracy: ", metrics.accuracy_score(y_test, y_pred))

In [ ]:
# Create the plot with a large enough figure
fig, ax = plt.subplots(figsize=(80, 40))

# Plot the 16th decision tree
plot_tree(
    clf3.estimators_[15],
    ax=ax,
    fontsize=20,
    feature_names=df_decade.columns,
    class_names=['unpleasant', 'pleasant'],
    filled=True,
    rounded=True
)

# Save the figure
fig.savefig(os.path.join(path, 'Visualizations', 'Random_Forest_10s_Optimized.png'), bbox_inches='tight')

# Show the figure
plt.show()

### Uncovering Feature Importance

In [ ]:
## Retrieve feature importances from the trained Random Forest
newarray = clf3.feature_importances_

print(newarray.shape)
newarray

In [ ]:
# Create a list of weather features

wx_list = [feature.replace('BUDAPEST_', '') for feature in BUDAPEST_list]
wx_list

In [ ]:
important = pd.Series(newarray, index = wx_list)
important

In [ ]:
plt.style.use('fivethirtyeight')

# List of x locations for plotting
x_values = list(range(len(newarray)))

# Debugging output (optional)
print(wx_list)

# Create the figure and plot
fig = plt.figure(figsize=(12, 6))  # Add a figure object with a reasonable size
plt.bar(x_values, newarray, orientation='vertical')
plt.xticks(x_values, wx_list, rotation='vertical')
plt.ylabel('Importance')
plt.xlabel('Feature')
plt.title('Feature Importances for BUDAPEST (all years)')

# Save the figure
plt.savefig(os.path.join(path, 'Visualizations', 'BUDAPEST_feature_importances_optimized.png'), bbox_inches='tight')
plt.show()